The project aims to create an application that will recommend destinations to Kayak users for their next vacation.

To make these recommendations, we need to scrape several data points:


1. **Geolocation**
- Latitude/longitude of each city studied

2. **Weather**
- Minimum/maximum temperature over 7 days
- Probability of rain
- Humidity
- Weather description

3. **Hotels**
- Hotel name
- Page URL
- Latitude/Longitude
- User rating
- Description

## 1. Gelocation

First and foremost, we will need to retrieve the geographic coordinates of the cities we have selected. This data will then be necessary for weather and hotel searches.            
We will use the Nominatim API for this              
[Nominatim](https://nominatim.org/)             
[Documentation](https://nominatim.org/release-docs/develop/api/Search/)

In [82]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import json
import pandas as pd
import boto3
from dotenv import load_dotenv
import plotly.express as px

In [3]:
# selected cities
cities = [
"Mont Saint Michel",
"St Malo",
"Bayeux",
"Le Havre",
"Rouen",
"Paris",
"Amiens",
"Lille",
"Strasbourg",
"Chateau du Haut Koenigsbourg",
"Colmar",
"Eguisheim",
"Besancon",
"Dijon",
"Annecy",
"Grenoble",
"Lyon",
"Gorges du Verdon",
"Bormes les Mimosas",
"Cassis",
"Marseille",
"Aix en Provence",
"Avignon",
"Uzes",
"Nimes",
"Aigues Mortes",
"Saintes Maries de la mer",
"Collioure",
"Carcassonne",
"Ariege",
"Toulouse",
"Montauban",
"Biarritz",
"Bayonne",
"La Rochelle"
]

In [4]:
# Function to collect coordinates
def get_coordinates(city_name):
    # Nominatim requires an identifiable User-Agent
    headers = {"User-Agent": "kayak-trip-planner/1.0"}
    params = {
        "q": city_name + ", France", # To search with string
        "format": "json", 
        "limit": 1 # Because we want one city at a time
    }

    response = requests.get(
        "https://nominatim.openstreetmap.org/search",
        headers=headers,
        params=params
    )

    data = response.json()
    if data:  # Because an empty list equals "False"
        return {
            "city": city_name,
            "latitude": float(data[0]["lat"]),
            "longitude": float(data[0]["lon"])
        }

    # If data is empty, we return NaN 
    return {"city": city_name, "latitude": None, "longitude": None}

In [7]:
# A for loop to collect data in JSON
results = []

for city in cities:
    coords = get_coordinates(city)
    results.append(coords)
    # Nominatim cannot have more than one request per second
    time.sleep(1)  

# Build dataset 
df_cities = pd.DataFrame(results)
df_cities

,city,latitude,longitude
0,Mont Saint Michel,48.635954,-1.511460
1,St Malo,48.649518,-2.026041
2,Bayeux,49.276462,-0.702474
3,Le Havre,49.493898,0.107973
4,Rouen,49.440459,1.093966
5,Paris,48.853495,2.348391
6,Amiens,49.894171,2.295695
7,Lille,50.636565,3.063528
8,Strasbourg,48.584614,7.750713
9,Chateau du Haut Koenigsbourg,48.249382,7.343941


We have now retrieved the coordinates for each of the cities. No row returns an empty value, so we can proceed.

## 2. Weather

To retrieve weather data, we will use the API OpenWeather.           
Here is the documentation for it:       
[Documentation OpenWeather](https://openweathermap.org/api/one-call-api)        
[How call data with OpenWeather](https://openweathermap.org/api/one-call-4?collection=one_call_api)   

We will thus retrieve the following data:         
- Minimum/maximum temperature over 7 days
- Probability of rain
- Humidity
- Weather description

According to the documentation, to retrieve data for a day you need to use: 
``` 
https://api.openweathermap.org/data/4.0/onecall/timeline/1day?lat=51.5&lon=-0.1&appid={API key}
```
and we can also see that the default temperature is in Kelvin. We can load this with: ``units: metric`` pour Celsius

In [ ]:
# We testing before to do the function 
# l'API 4.0 doesn't works we our subscription 
# So we need to use the 2.5 version
response = requests.get(
    "https://api.openweathermap.org/data/2.5/forecast",
    params={
        "lat": 48.8566,
        "lon": 2.3522,
        "units": "metric",
        "appid": API_KEY
    }
)
data = response.json()
print(data)

{'cod': '200', 'message': 0, 'cnt': 40, 'list': [{'dt': 1780498800, 'main': {'temp': 19.64, 'feels_like': 19.33, 'temp_min': 19.64, 'temp_max': 19.95, 'pressure': 1011, 'sea_level': 1011, 'grnd_level': 1000, 'humidity': 64, 'temp_kf': -0.31}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'clouds': {'all': 97}, 'wind': {'speed': 6.21, 'deg': 213, 'gust': 13.04}, 'visibility': 10000, 'pop': 0, 'sys': {'pod': 'd'}, 'dt_txt': '2026-06-03 15:00:00'}, {'dt': 1780509600, 'main': {'temp': 18.2, 'feels_like': 18.06, 'temp_min': 17.56, 'temp_max': 18.2, 'pressure': 1009, 'sea_level': 1009, 'grnd_level': 998, 'humidity': 76, 'temp_kf': 0.64}, 'weather': [{'id': 500, 'main': 'Rain', 'description': 'light rain', 'icon': '10d'}], 'clouds': {'all': 99}, 'wind': {'speed': 6.79, 'deg': 214, 'gust': 15.04}, 'visibility': 10000, 'pop': 0.2, 'rain': {'3h': 0.13}, 'sys': {'pod': 'd'}, 'dt_txt': '2026-06-03 18:00:00'}, {'dt': 1780520400, 'main': {'temp': 19.24,

In [ ]:

load_dotenv()
API_KEY = os.getenv("OPENWEATHER_API_KEY")

# Definition function
def get_weather(lat, lon, city_name):
    url = "https://api.openweathermap.org/data/2.5/forecast"
    params = {
        "lat": lat,
        "lon": lon,
        "units": "metric",
        "appid": API_KEY
    }
    response = requests.get(url, params=params)
    data = response.json()
    
    # On met les 40 tranches dans un DataFrame
    rows = []
    for entry in data["list"]:
        rows.append({
            "date": entry["dt_txt"][:10],  
            "temp": entry["main"]["temp"],
            "humidity": entry["main"]["humidity"],
            "pop": entry["pop"],
            "description": entry["weather"][0]["description"]
        })
    
    df = pd.DataFrame(rows)
    
    # Group by day 
    df_daily = df.groupby("date").agg(
        temp_min=("temp", "min"),
        temp_max=("temp", "max"),
        humidity=("humidity", "mean"),
        pop=("pop", "max"),
        description=("description", "first")
    ).reset_index()
    
    df_daily["city"] = city_name
    return df_daily

In [19]:
# Testing with one city
df_paris = get_weather(48.8566, 2.3522, "Paris")
df_paris

,date,temp_min,temp_max,humidity,pop,description,city
0,2026-06-03,18.20,19.64,74.666667,0.32,overcast clouds,Paris
1,2026-06-04,15.31,22.84,71.000000,1.00,light rain,Paris
2,2026-06-05,11.34,20.62,54.250000,0.00,broken clouds,Paris
3,2026-06-06,15.33,21.46,55.500000,1.00,overcast clouds,Paris
4,2026-06-07,13.52,24.38,55.000000,0.00,broken clouds,Paris
5,2026-06-08,17.21,24.02,59.600000,0.20,light rain,Paris


We can't use our subscription plan for have access to more of 6 days, which is sufficient.

In [20]:
# This works, so we can use the function for all selected cities
all_weather = []
for _, row in df_cities.iterrows():  # _ for index
    df_city = get_weather(row["latitude"], row["longitude"], row["city"])
    all_weather.append(df_city)

df_weather = pd.concat(all_weather, ignore_index=True)
df_weather

,date,temp_min,temp_max,humidity,pop,description,city
0,2026-06-03,15.04,17.05,84.000,1.00,light rain,Mont Saint Michel
1,2026-06-04,11.62,16.26,80.375,1.00,light rain,Mont Saint Michel
2,2026-06-05,8.63,17.07,73.000,0.51,light rain,Mont Saint Michel
3,2026-06-06,12.63,16.51,79.250,1.00,overcast clouds,Mont Saint Michel
4,2026-06-07,9.73,20.93,76.750,0.00,scattered clouds,Mont Saint Michel
...,...,...,...,...,...,...,...
205,2026-06-04,15.94,17.40,70.250,1.00,light rain,La Rochelle
206,2026-06-05,14.71,17.22,64.125,1.00,light rain,La Rochelle
207,2026-06-06,14.33,18.58,69.250,1.00,broken clouds,La Rochelle
208,2026-06-07,15.86,19.08,68.250,0.20,scattered clouds,La Rochelle


In [21]:
# Saving
df_weather.to_csv("weather_6_days.csv", index=False)

# 3. Hotels

Now we move on to hotels. We will therefore scrape the information from booking.com.

We will retrieve the following information:
- Hotel name
- Page URL
- Latitude/Longitude
- User rating
- Description

In [ ]:
# Testint to access to Booking.com
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

url = "https://www.booking.com/searchresults.html?ss=Paris&lang=en-gb"

response = requests.get(url, headers=headers)
print(response.status_code)

202


Code 202 indicates that everything went smoothly. However, let's see if we can get a real HTML response without an anti-scraping system getting involved.

In [23]:
print(response.text[:500])

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title></title>
    <style>
        body {
            font-family: "Arial";
        }
    </style>
    <script type="text/javascript" nonce="3kDPjM07SjrWm9S">
        function getAjaxObject() {
    var ajax;
    if (window.XMLHttpRequest) {
        try {
            ajax = new window.XMLHttpRequest(); // XMLHttpRequest (Mozilla, Opera, S


Booking returns a page containing JavaScript. This JavaScript must execute in the browser for us to see the actual content. This is a standard anti-scraping protection.

We will therefore attempt to scrape using Selenium. This will simulate a real browser that executes the JavaScript.

In [29]:
# Driver configuration
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Chrome without a graphical interface
options.add_argument("--no-sandbox") # Necessary for certain Docker environments
options.add_argument("--disable-dev-shm-usage") # Avoids memory issues

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()), # install driver
    options=options  # With our options 
)

In [30]:
# Testing with Paris
url = "https://www.booking.com/searchresults.html?ss=Paris&lang=en-gb"
driver.get(url)

print(driver.title)
print(driver.page_source[:500])


<html lang="en"><head>
        <meta charset="utf-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title></title>
        <style>
            body {
                font-family: "Arial";
            }
        </style>
        <script type="text/javascript" nonce="17805010705980.7163415359338294">
            function getAjaxObject() {
                var ajax;
                if (window.XMLHttpRequest) {
                    try {
                        a


We still have a JavaScript page. This is either due to a protection measure or a timeout issue before the download starts. Indeed, we first have a JavaScript page, which then executes to return the HTML page. Perhaps we were simply too quick. Let's say a timeout of 5 seconds.

In [ ]:
import time

driver.get(url)
time.sleep(5)  # wait 5 secondes

print(driver.title)
print(driver.page_source[:1000])

Booking.com: Search results. Book your hotel now!
<html lang="en-gb" prefix="og: http://ogp.me/ns# fb: http://ogp.me/ns/fb# booking_com: http://ogp.me/ns/fb/booking_com#" class=" b_bot b_bot supports_fontface supports_hyphens  hasJS" dir="ltr"><head profile="http://a9.com/-/spec/opensearch/1.1/">
<script async="" type="text/plain" src="https://www.googletagmanager.com/gtm.js?id=GTM-5Q664QZ" class="optanon-category-C0002-C0004 "></script><script type="text/javascript" nonce="" src="https://cdn.cookielaw.org/consent/3ea94870-d4b1-483a-b1d2-faf1d982bb31/OtAutoBlock.js"></script>
<script type="text/javascript" nonce="">
(function () {
document.addEventListener('click', function(e) {
if (e.target && e.target.classList.contains('ot-preference-center-footer')) {
e.preventDefault();
Optanon && Optanon.ToggleInfoDisplay();
}
});
document.addEventListener('cookie_banner_closed', function(e) {
if (window.PCM && window.B && window.B.et) {
window.B.et.goal((window.PCM.Marketing || window.PCM.Analyt

It worked, so we can now begin to look at how the information that interests us is written.

By inspecting the booking.com page, we can see:

- the hotel name is in this part of the code: `<div data-testid="title" class="b87c397a13 a3e0b4ffd1">Hotel Lucien</div>`

- the score: `<div aria-hidden="true" class="f63b14ab7a dff2e52086">8.2</div>`

- Link to hotel : `<a href="https://www.booking.com/hotel/fr/imperial-paris.en-gb.html?aid=304142&amp;label=gen173nr-10CAQoggJCDHNlYXJjaF9wYXJpc0gJWARoTYgBAZgBM7gBF8gBDNgBA-gBAfgBAYgCAagCAbgCjZeB0QbAAgHSAiQyMjBkNzUyNS0yNGVjLTQ5YmMtYjgxNC1jNDUzNDhlNzg4YmTYAgHgAgE&amp;ucfs=1&amp;arphpl=1&amp;checkin=2026-06-03&amp;checkout=2026-06-04&amp;group_adults=1&amp;req_adults=1&amp;no_rooms=1&amp;group_children=0&amp;req_children=0&amp;hpos=2&amp;hapos=2&amp;sr_order=popularity&amp;srpvid=84286e865aea0415&amp;srepoch=1780501747&amp;all_sr_blocks=118949901_424416955_0_2_0&amp;highlighted_blocks=118949901_424416955_0_2_0&amp;matching_block_id=118949901_424416955_0_2_0&amp;sr_pri_blocks=118949901_424416955_0_2_0__18053&amp;from=searchresults" class="bd77474a8e"`
- the description is located on another page. Two-level scraping is required.

On the hotel's page:
- The description : `<p data-testid="property-description" class="b99b6ef58f f1152bae71">Situé à Paris, l'Hotel Imperial Paris est un établissement 3&nbsp;étoiles doté d'une connexion Wi-Fi gratuite dans tous ses locaux. L’établissement se trouve à 500&nbsp;mètres de l'Opéra Garnier et à 900&nbsp;mètres de la gare Saint-Lazare.
Toutes les chambres, climatisées, disposent d’une télévision à écran plat et d’un bureau. Leur salle de bains privative est pourvue d’un sèche-cheveux et d’articles de toilette gratuits.
L'Hotel Imperial Paris possède une réception ouverte 24h/24&nbsp;et une bagagerie.
L'hôtel vous place à 230&nbsp;mètres de la station de métro Notre-Dame-de-Lorette (ligne 12), qui vous permettra de rejoindre directement le quartier de Montmartre et la place de la Concorde.
Cet établissement accepte les paiements par carte de débit/crédit.</p>`

- GPS data: ``booking.env.b_map_center_latitude = 48.87507333;
booking.env.b_map_center_longitude = 2.33613217;``

In [38]:
# Our scapping function
def get_hotels_list(city_name):
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    
    url = f"https://www.booking.com/searchresults.html?ss={city_name}&lang=en-gb"
    driver.get(url)
    
    # This didn't work with a simple sleep(5)
    # So we'll wait until the results are fully loaded before retrieving them
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="title"]'))
        )
    # to avoid an infinite loop  
    except:
        print("Timeout: item not found")
        print(driver.page_source[:1000])
        driver.quit()
        return []

    # We create a list for our dictionaries
    hotels = []
    
    # We take the names 
    names = driver.find_elements(By.CSS_SELECTOR, '[data-testid="title"]')
    # We take ce rates
    scores = driver.find_elements(By.CSS_SELECTOR, '.f63b14ab7a.dff2e52086')
    # We take the links
    links = driver.find_elements(By.CSS_SELECTOR, '[data-testid="title-link"]')
    
    # We create a dictionnary with this informations
    for i in range(len(names)):
        hotels.append({
            "city": city_name,
            "name": names[i].text,
            "score": scores[i].text if i < len(scores) else None,
            "url": links[i].get_attribute("href").split("?")[0]
        })
    
    driver.quit()
    return hotels


In [39]:
# Testing with Paris 
hotels_paris = get_hotels_list("Paris")
print(f"{len(hotels_paris)} hôtels found")
print(hotels_paris[0])

Timeout: item not found
<html lang="en"><head>
        <meta charset="utf-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title></title>
        <style>
            body {
                font-family: "Arial";
            }
        </style>
        <script type="text/javascript" nonce="17805029429940.09812161846950163">
            function getAjaxObject() {
                var ajax;
                if (window.XMLHttpRequest) {
                    try {
                        ajax = new window.XMLHttpRequest(); // XMLHttpRequest (Mozilla, Opera, Safari, etc.)
                    } catch (e) {
                        return false;
                    }
                } else {
                    var msXML = new Array( // XMLHttpRequest (IE with ActiveX)
                        "Msxml2.XMLHTTP.5.0",
                        "Msxml2.XMLHTTP.4.0",
                        "Msxml2.XMLHTTP.3.0",
                        "Msxml2.XMLHTTP",
             

IndexError: list index out of range

Booking detects Selenium in headless mode and blocks the request. This is where we see real protection against scraping.
We'll try without headless mode.

In [40]:
def get_hotels_list(city_name):
    options = webdriver.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    
    url = f"https://www.booking.com/searchresults.html?ss={city_name}&lang=en-gb"
    driver.get(url)
    time.sleep(8)  # We wait more time
    
    print(driver.title)
    names = driver.find_elements(By.CSS_SELECTOR, '[data-testid="title"]')
    print(f"{len(names)} hotel found")
    
    driver.quit()

get_hotels_list("Paris")

Booking.com: Hotels in Paris. Book your hotel now!
25 hotel found


It works the headless connection was indeed a problem. Now we can retrieve all the information.

In [ ]:
# The same function but without headless and with the addition of a user agent
def get_hotels_list(city_name):
    options = webdriver.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    
    url = f"https://www.booking.com/searchresults.html?ss={city_name}&lang=en-gb"
    driver.get(url)
    time.sleep(8)
    
    names = driver.find_elements(By.CSS_SELECTOR, '[data-testid="title"]')
    scores = driver.find_elements(By.CSS_SELECTOR, '.f63b14ab7a.dff2e52086')
    links = driver.find_elements(By.CSS_SELECTOR, '[data-testid="title-link"]')
    
    hotels = []
    for i in range(len(names)):
        hotels.append({
            "city": city_name,
            "name": names[i].text,
            "score": scores[i].text if i < len(scores) else None,
            "url": links[i].get_attribute("href").split("?")[0]
        })
    
    driver.quit()
    return hotels

hotels_paris = get_hotels_list("Paris")
print(f"{len(hotels_paris)} hotel found")
print(hotels_paris[0])

25 hotel found
{'city': 'Paris', 'name': 'JOIVY Premium flats in the heart of the 9th arr, near Moulin Rouge', 'score': '', 'url': 'https://www.booking.com/hotel/fr/altido-pleasant-1-bed-flat-with-shared-courtyard.en-gb.html'}


In [42]:
print(hotels_paris[1])
print(hotels_paris[2])
print(hotels_paris[3])

{'city': 'Paris', 'name': 'Hotel Meslay Republique', 'score': '', 'url': 'https://www.booking.com/hotel/fr/meslay-republique.en-gb.html'}
{'city': 'Paris', 'name': '', 'score': '', 'url': 'https://www.booking.com/hotel/fr/bradfordelysees.en-gb.html'}
{'city': 'Paris', 'name': '', 'score': '', 'url': 'https://www.booking.com/hotel/fr/paris-rivoli-notre-dame.en-gb.html'}


The scores and hotel names are not displayed.
We will first look at the box containing the hotel information.

In [ ]:
options = webdriver.ChromeOptions()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

url = "https://www.booking.com/searchresults.html?ss=Paris&lang=en-gb"
driver.get(url)
time.sleep(8)

# We take the card
cards = driver.find_elements(By.CSS_SELECTOR, '[data-testid="property-card"]')
print(f"{len(cards)} cartes trouvées")
print(cards[0].get_attribute("innerHTML")[:500])

25 cartes trouvées
<div class="aa97d6032f" data-testid="property-card-container"><div class="e05069daf3"><div class="c17271c4d7"><a href="https://www.booking.com/hotel/fr/altido-pleasant-1-bed-flat-with-shared-courtyard.en-gb.html?aid=304142&amp;label=gen173nr-10CAQoggJCDHNlYXJjaF9wYXJpc0gJWARoTYgBAZgBM7gBF8gBDNgBA-gBAfgBAYgCAagCAbgCy6aB0QbAAgHSAiQxNzhiYWVkYS05MDczLTRlY2EtYjQ3Mi03ODQ2MTRmNWFiYTHYAgHgAgE&amp;ucfs=1&amp;arphpl=1&amp;group_adults=2&amp;req_adults=2&amp;no_rooms=1&amp;group_children=0&amp;req_children


It works. Now we'll try to extract each piece of information from this card.

In [45]:
hotels = []
for card in cards:
    try:
        name = card.find_element(By.CSS_SELECTOR, '[data-testid="title"]').text
    except:
        name = None
    
    try:
        score = card.find_element(By.CSS_SELECTOR, '.f63b14ab7a.dff2e52086').text
    except:
        score = None
    
    try:
        url = card.find_element(By.CSS_SELECTOR, '[data-testid="title-link"]').get_attribute("href").split("?")[0]
    except:
        url = None
    
    hotels.append({
        "name": name,
        "score": score,
        "url": url
    })

print(hotels[0])
print(hotels[1])
print(hotels[2])

{'name': None, 'score': None, 'url': None}
{'name': None, 'score': None, 'url': None}
{'name': None, 'score': None, 'url': None}


That doesn't work. We'll try to retrieve the HTML directly from the card when we retrieve the card.

In [ ]:
options = webdriver.ChromeOptions()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

url = "https://www.booking.com/searchresults.html?ss=Paris&lang=en-gb"
driver.get(url)
time.sleep(8)

# We take the card 
# And we take all the HTML of this card
soup = BeautifulSoup(driver.page_source, "html.parser")
cards = soup.find_all(attrs={"data-testid": "property-card"})
print(f"{len(cards)} cartes trouvées")
print(str(cards[0])[:1000])

25 cartes trouvées
<div aria-label="Property" class="c3bdfd4ac2 a0ab5da06c d46ff48a92 f728e61e72 d0acd69e66 c256f1a28a bc2204a477" data-testid="property-card" role="listitem" style="--bui_box_spaced_padding--s: 4;"><div class="aa97d6032f" data-testid="property-card-container"><div class="e05069daf3"><div class="c17271c4d7"><a aria-hidden="true" data-testid="property-card-desktop-single-image" href="https://www.booking.com/hotel/fr/meslay-republique.en-gb.html?aid=304142&amp;label=gen173nr-10CAQoggJCDHNlYXJjaF9wYXJpc0gJWARoTYgBAZgBM7gBF8gBDNgBA-gBAfgBAYgCAagCAbgCwaiB0QbAAgHSAiQ0MTg0M2IxNS1kMDBjLTQ5MTctOTIzMy00M2UzMDc2ODBlMzPYAgHgAgE&amp;ucfs=1&amp;arphpl=1&amp;group_adults=2&amp;req_adults=2&amp;no_rooms=1&amp;group_children=0&amp;req_children=0&amp;hpos=1&amp;hapos=1&amp;sr_order=popularity&amp;srpvid=679272e0654c02bc&amp;srepoch=1780503621&amp;from=searchresults" rel="noopener noreferrer" tabindex="-1" target="_blank"><img alt="Hotel Meslay Republique" class="eedb7a060f" data-testid="

It works, we have the HTML of the map. Now let's try to extract the information we're interested in from it.

In [48]:
card = cards[0]

# Name
name = card.find(attrs={"data-testid": "title"})
print("Nom:", name.text if name else None)

# rate
score = card.find(class_="f63b14ab7a dff2e52086")
print("Score:", score.text if score else None)

# URL
link = card.find(attrs={"data-testid": "title-link"})
print("URL:", link["href"].split("?")[0] if link else None)

Nom: Hotel Meslay Republique
Score: 8.3
URL: https://www.booking.com/hotel/fr/meslay-republique.en-gb.html


It works. We can therefore develop our function further so that it works correctly.

In [49]:
def get_hotels_list(city_name):
    options = webdriver.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    
    url = f"https://www.booking.com/searchresults.html?ss={city_name}&lang=en-gb"
    driver.get(url)
    time.sleep(8)
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()
    
    cards = soup.find_all(attrs={"data-testid": "property-card"})
    
    hotels = []
    for card in cards:
        name = card.find(attrs={"data-testid": "title"})
        score = card.find(class_="f63b14ab7a dff2e52086")
        link = card.find(attrs={"data-testid": "title-link"})
        
        hotels.append({
            "city": city_name,
            "name": name.text if name else None,
            "score": score.text if score else None,
            "url": link["href"].split("?")[0] if link else None
        })
    
    return hotels


In [50]:
# Testing on Paris 
hotels_paris = get_hotels_list("Paris")
print(f"{len(hotels_paris)} hôtels trouvés")
print(hotels_paris[0])

25 hôtels trouvés
{'city': 'Paris', 'name': 'Hotel Meslay Republique', 'score': '8.3', 'url': 'https://www.booking.com/hotel/fr/meslay-republique.en-gb.html'}


It works.
Now we create a new function for the second page. We just need to slightly modify the first function.

In [ ]:
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

url = "https://www.booking.com/hotel/fr/meslay-republique.en-gb.html"
driver.get(url)
time.sleep(8)

page_source = driver.page_source  # saving
driver.quit()                      

soup = BeautifulSoup(page_source, "html.parser")

# Description
description = soup.find(attrs={"data-testid": "property-description"})
print("Description:", description.text[:100] if description else None)

# GPS
import re
lat = re.search(r'b_map_center_latitude\s*=\s*([\d.]+)', page_source)
lon = re.search(r'b_map_center_longitude\s*=\s*([\d.]+)', page_source)
print("Lat:", lat.group(1) if lat else None)
print("Lon:", lon.group(1) if lon else None)

Description: Doté d'une connexion Wi-Fi gratuite dans tout l'établissement, l'Hotel Meslay Republique se situe à 
Lat: 48.86730056
Lon: 2.36223970


It works, so we can do it as a function.

In [53]:
def get_hotel_details(url):
    options = webdriver.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    
    driver.get(url)
    time.sleep(8)
    
    page_source = driver.page_source
    driver.quit()
    
    soup = BeautifulSoup(page_source, "html.parser")
    
    description = soup.find(attrs={"data-testid": "property-description"})
    lat = re.search(r'b_map_center_latitude\s*=\s*([\d.]+)', page_source)
    lon = re.search(r'b_map_center_longitude\s*=\s*([\d.]+)', page_source)
    
    return {
        "description": description.text if description else None,
        "latitude": float(lat.group(1)) if lat else None,
        "longitude": float(lon.group(1)) if lon else None
    }

# Test
details = get_hotel_details("https://www.booking.com/hotel/fr/meslay-republique.en-gb.html")
print(details)

{'description': "Doté d'une connexion Wi-Fi gratuite dans tout l'établissement, l'Hotel Meslay Republique se situe à Paris, à 100 mètres de la place de la République.\n\nToutes les chambres comportent une télévision par satellite à écran plat. Leur salle de bains est pourvue d'articles de toilette gratuits et d'un sèche-cheveux.\n\nLa réception de l'établissement est ouverte 24h/24.\n\nL'Hotel Meslay République se trouve à 1,5 km du centre Pompidou et à moins de 2 km de la cathédrale Notre-Dame. L'aéroport d'Orly est implanté à 20 km.", 'latitude': 48.86730056, 'longitude': 2.3622397}


Perfect, now we're going to put the two together.

In [ ]:
# Test before to do the booking_scraper.py
all_hotels = []

# We're checking if we can properly use an element like the URL
for hotel in hotels_paris:
    if hotel["url"]:
        details = get_hotel_details(hotel["url"])
        hotel.update(details)  # We merge the both in one dictionary
    all_hotels.append(hotel)

# We're testing if we can create a dataset with this.
df_hotels = pd.DataFrame(all_hotels)
df_hotels.head()

It works. After this research phase, we can now create our booking_scraper.py file.

We then started the scraping with ``booking_scraper.py``. We will now convert the JSON to CVS.

In [ ]:
# We take our file from the booking_scraper.py
with open("hotels_raw.json", "r", encoding="utf-8") as f:
    content = f.read()

# Expression that all JSON objects search for in the text
objects = re.findall(r'\{[^{}]+\}', content, re.DOTALL)

hotels = []
# We loop through each piece of JSON found
for obj in objects:
    try:
        hotels.append(json.loads(obj))
    except:
        pass

# We can now transform it into a dataframe
df_hotels = pd.DataFrame(hotels)
print(df_hotels.shape)
df_hotels.head()

(800, 7)


,city,name,score,url,description,latitude,longitude
0,Le Havre,Studio Meublé Vue Mer,6.5,https://www.booking.com/hotel/fr/studio-meuble...,Proposant un casino et offrant une vue sur la ...,49.488946,0.099551
1,Rouen,Les Matelots Rouen Centre by MyFrenchPAT,8.9,https://www.booking.com/hotel/fr/le-diplomatic...,L’hébergement Les Matelots Rouen Centre by MyF...,49.447545,1.101662
2,Paris,ibis Paris Bastille Opera,8.0,https://www.booking.com/hotel/fr/paris-bastill...,"Situé dans le centre-ville, à 550 mètres de la...",48.857373,2.372880
3,Paris,Timhotel Odessa Montparnasse,7.9,https://www.booking.com/hotel/fr/le-michel-mon...,Le Timhotel Odessa Montparnasse est situé au c...,48.842689,2.324306
4,Paris,Hotel de Seze,8.9,https://www.booking.com/hotel/fr/de-seze-paris...,Situé à 50 mètres de l'église de la Madeleine ...,48.870575,2.326384


In [ ]:
# Saving
df_hotels.to_csv("hotels_raw.csv", index=False)
print(f"saving. {len(df_hotels)} hotels")

Now we can send this file to our S3 Bucket

In [ ]:
# Load .env
load_dotenv(
    dotenv_path=r"C:\Users\axelv\Documents\Jedha\DATA_SCIENCE\LEAD\Projet\Kayak\.env",
    override=True
)

# Credentials
s3 = boto3.client(
    "s3",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name="eu-west-3"
)

# Log with Boto3
s3 = boto3.client("s3")

BUCKET_NAME = "kayak-av-jedha"

# we convert df_weather to csv 
df_weather.to_csv("weather_raw.csv", index=False)
s3.upload_file("weather_raw.csv", BUCKET_NAME, "weather_raw.csv")
print("weather_raw.csv uploaded")

s3.upload_file("hotels_raw.csv", BUCKET_NAME, "hotels_raw.csv")
print("hotels_raw.csv uploaded")

weather_raw.csv uploaded
hotels_raw.csv uploaded


## 4. Weather scoring

Now we're going to give cities a score in order to recommend the five best destinations.

We chose three criteria for scoring:

- Temperature
- Absence of rain
- Humidity

To calculate the score, we will add up the ranking of the cities for each of these variables.

In [75]:
# Loading weather csv 
df_weather = pd.read_csv("weather_raw.csv")
df_weather.head()

,date,temp_min,temp_max,humidity,pop,description,city
0,2026-06-03,15.04,17.05,84.000,1.00,light rain,Mont Saint Michel
1,2026-06-04,11.62,16.26,80.375,1.00,light rain,Mont Saint Michel
2,2026-06-05,8.63,17.07,73.000,0.51,light rain,Mont Saint Michel
3,2026-06-06,12.63,16.51,79.250,1.00,overcast clouds,Mont Saint Michel
4,2026-06-07,9.73,20.93,76.750,0.00,scattered clouds,Mont Saint Michel


In [76]:
# we calculate the average for each city
df_score = df_weather.groupby("city").agg(
    avg_temp_max=("temp_max", "mean"),
    avg_temp_min=("temp_min", "mean"),
    avg_humidity=("humidity", "mean"),
    avg_pop=("pop", "mean")
).reset_index()

df_score.head()

,city,avg_temp_max,avg_temp_min,avg_humidity,avg_pop
0,Aigues Mortes,23.841667,18.003333,65.454167,0.300000
1,Aix en Provence,26.125000,18.256667,51.131944,0.333333
2,Amiens,19.121667,12.743333,75.638889,0.700000
3,Annecy,23.628333,13.855000,70.880556,0.466667
4,Ariege,20.051667,11.971667,76.470833,0.386667


In [77]:
# The data is ranked to create the score.
df_score["score"] = (
    df_score["avg_temp_max"].rank() +           
    df_score["avg_humidity"].rank(ascending=False) +  
    df_score["avg_pop"].rank(ascending=False)  
)

In [80]:
# Top 5
top_5 = df_score.sort_values("score", ascending=False).head(5)
print(top5[["city", "score"]])

                  city  score
5              Avignon   99.5
27               Nimes   95.5
1      Aix en Provence   94.5
10  Bormes les Mimosas   94.0
0        Aigues Mortes   85.5


## 5. Plotly diagrams
### 5.1. Weather plotly

We will now create the plotly diagrams here, which we will then use in the streamlit.

In [ ]:
top_5_cities = top_5["city"].tolist()
df_top_5 = df_cities[df_cities["city"].isin(top_5_cities)].merge(top_5[["city", "score"]], on="city")

# Plotly
fig = px.scatter_mapbox(
    df_top_5,
    lat="latitude",
    lon="longitude",
    hover_name="city",
    size="score",
    color="score",
    zoom=4,
    title="Top 5 weather destinations in France"
)

fig.update_layout(mapbox_style="open-street-map")
fig.show()

C:\Users\axelv\AppData\Local\Temp\ipykernel_32744\4271641362.py:5: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



### 5.2. Hotel plotly

Now let's do the same with hotels

In [ ]:
df_top20_hotels = df_hotels[df_hotels["city"].isin(top_5_cities)].copy()

# We convert the score to a float and sort it
df_top20_hotels["score"] = pd.to_numeric(df_top20_hotels["score"], errors="coerce")
df_top20_hotels = df_top20_hotels.sort_values("score", ascending=False).head(20)

# Plotly
fig2 = px.scatter_mapbox(
    df_top20_hotels,
    lat="latitude",
    lon="longitude",
    hover_name="name",
    hover_data={"score": True, "city": True},
    size="score",
    color="score",
    zoom=4,
    title="Top 20 hôtels dans les meilleures destinations"
)

fig2.update_layout(mapbox_style="open-street-map")
fig2.show()

C:\Users\axelv\AppData\Local\Temp\ipykernel_32744\2513892712.py:8: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



Now that we have tested these plotly schemes here, we can do the streamlit

We will use the interactive aspect of Streamlit to allow users to choose the importance of criteria such as temperature, absence of rain, or absence of humidity.